In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from transformers import pipeline

from nltk import sent_tokenize
import nltk
from time import time
import matplotlib.pyplot as plt
import torch
import re
import os

In [105]:
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\james\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Load Model

In [106]:
model_name = "facebook/bart-large-mnli"
device = 0 if torch.cuda.is_available() else -1 # Use GPU if available

In [107]:
device

-1

In [108]:
def load_model(device):
    """
    Load the BART model for NLI.
    """
    theme_classifier = pipeline(
        "zero-shot-classification", 
        model=model_name, 
        device=device)
    return theme_classifier

In [109]:
theme_classifier = load_model(device)

Device set to use cpu


In [110]:
theme_list = "Friendship, Stupidity, Ignorance, Family, Cult, Love, Betrayal, Revenge, Justice, Sacrifice, Courage, Loyalty, Greed, Power, Corruption, Redemption, Identity, Belonging, Freedom, Conformity".split(", ")

In [111]:
theme_classifier(
    "I drive better when I'm drunk. I can't care what no scientist says.",
    candidate_labels=theme_list,
    multi_label=True
)

{'sequence': "I drive better when I'm drunk. I can't care what no scientist says.",
 'labels': ['Stupidity',
  'Power',
  'Family',
  'Belonging',
  'Betrayal',
  'Conformity',
  'Greed',
  'Identity',
  'Freedom',
  'Cult',
  'Redemption',
  'Sacrifice',
  'Ignorance',
  'Revenge',
  'Corruption',
  'Courage',
  'Loyalty',
  'Justice',
  'Friendship',
  'Love'],
 'scores': [0.5023714303970337,
  0.4621305763721466,
  0.33699411153793335,
  0.3022654354572296,
  0.21013042330741882,
  0.195215106010437,
  0.18947237730026245,
  0.1751503348350525,
  0.10232897102832794,
  0.09535229206085205,
  0.09192289412021637,
  0.07779225707054138,
  0.07004879415035248,
  0.0664195716381073,
  0.0629066675901413,
  0.059819430112838745,
  0.017386021092534065,
  0.015693364664912224,
  0.007004039827734232,
  0.0021328593138605356]}

# Load Dataset

In [112]:
def parse_srt(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        with open(filename, 'r', encoding='latin-1') as f:
            lines = f.readlines()

    blocks = []
    block = []
    for line in lines:
        line = line.strip()
        if line == '':
            if block:
                blocks.append(block)
                block = []
        else:
            block.append(line)
    if block:
        blocks.append(block)

    sentences = []
    for block in blocks:
        text_lines = [
            l for l in block
            if not re.match(r'^\d+$', l)
            and not re.match(r'^\d{2}:\d{2}:\d{2},\d{3}', l)
            and '♪' not in l
            and not re.search(r'<.*?>', l)  # Remove lines with HTML tags
            and not re.search(r'www\.', l, re.IGNORECASE)  # Remove lines with URLs
        ]
        text_lines = [re.sub(r'{\\an\d+}', '', l) for l in text_lines]
        if text_lines:
            sentences.append(' '.join(text_lines))
    return sentences

In [113]:
def get_season_episode_code(filename):
    # Try SxxEyy format first
    match = re.search(r'[Ss](\d{2})[Ee](\d{2})', filename)
    if match:
        return f"{match.group(1)}{match.group(2)}"
    # Try xxXyy format (e.g., 20x01)
    match = re.search(r'(\d{2})[xX](\d{2})', filename)
    if match:
        return f"{match.group(1)}{match.group(2)}"
    return None  # or handle missing code as needed

# Example usage:
files = glob("../data/subtitles/*.srt")
codes = [get_season_episode_code(os.path.basename(f)) for f in files]
codes = [c for c in codes if c is not None]
print(codes.__len__()   )

794


In [120]:
def load_subtitles_dataset(dataset_path):
    subtitles_paths = glob(dataset_path + "/*.srt")

    print(subtitles_paths)
    scripts = []
    episode_num = []

    for path in subtitles_paths:
        if not os.path.exists(path):
            print(f"File not found: {path}")
            continue
        try:
            sentences = parse_srt(path)
            if sentences:
                print(f"Loaded {len(sentences)} sentences from {path}")

                script = " ".join(sentences).strip()
                season_episode = get_season_episode_code(os.path.basename(path))

                scripts.append(script)
                episode_num.append(season_episode)
                
        
            else:
                print(f"No valid sentences found in {path}")
        except Exception as e:
            print(f"Error processing {path}: {e}")
    
    df = pd.DataFrame.from_dict({
                    "episode": episode_num,
                    "script": scripts
                })
    return df

In [121]:
dataset_path = "../data/subtitles"
df = load_subtitles_dataset(dataset_path)

['../data/subtitles\\Simpsons 20x01 Sex Pies and Idiot Scrapes.srt', '../data/subtitles\\Simpsons 20x02 Lost Verizon.srt', '../data/subtitles\\Simpsons 20x03 Double Double Boy in Trouble.srt', '../data/subtitles\\Simpsons 20x04 Treehouse of Horror XIX.srt', '../data/subtitles\\Simpsons 20x05 Dangerous Curves.srt', '../data/subtitles\\Simpsons 20x06 Homer and Lisa Exchange Cross Words.srt', '../data/subtitles\\Simpsons 20x07 Mypods and Broomsticks.srt', '../data/subtitles\\Simpsons 20x08 The Burns and the Bees.srt', '../data/subtitles\\Simpsons 20x09 Lisa the Drama Queen.srt', '../data/subtitles\\Simpsons 20x10 Take My Life Please.srt', '../data/subtitles\\Simpsons 20x11 How the Test Was Won.srt', '../data/subtitles\\Simpsons 20x12 No Loan Again Naturally.srt', '../data/subtitles\\Simpsons 20x13 Gone Maggie Gone.srt', '../data/subtitles\\Simpsons 20x14 In the Name of the Grandfather.srt', '../data/subtitles\\Simpsons 20x15 Wedding for Disaster.srt', '../data/subtitles\\Simpsons 20x16 Ee

In [124]:
df.head()

,episode,script
0,2001,The Simpsons D'oh! BART: Woo-hoo! St. Patrick'...
1,2002,"D'oh! BART: Ay, caramba! Out of gas? But how? ..."
2,2003,"D'oh! Stupid shopping list, turning food into ..."
3,2004,"Hello. I'd like to vote for president, governo..."
4,2005,D'oh! BILL: Bill and Marty here in the middle ...


# Run Model

In [130]:
script = df.iloc[200]['script']

In [132]:
print(script)

(chalk screeches) (bell rings) (work whistle blows) -(sucking) -(register beeping) (jazzy solo) (tires screech) (tires screeching) (horn honks) (tires screech) D'oh! -Aah! -(tires screech) MARTY: MARTY: BILL: You played the wrong record, didn't you? MARTY: Bah! This is just another Hallmark holiday cooked up to sell cards. Oh, a Valentine from my granddaughter. Could I have the envelope? "To Moe, from your secret admirer." Yoo-hoo! Oh, God, no. (kisses, belches) And this is for my huggy bug in honor of this special day. (thinking) Happy Valentine's Day. Oh, thank you, dear. Woo-hoo! I'll bet you have a lot of things planned, eh, Dad? Uh, kind of. Nobody makes a big deal out of Valentine's Day. Hey, Dad, why don't you give Mom her present? What a nice idea, son. It's, uh... upstairs on the second floor of our house. I'll just go get it. (footsteps) (door slams) (footsteps) (Homer grunting and groaning) (screaming) Apu, you got to help me. I need a Valentine's gift for my wife. Perhaps t